# 02 — Labeling

This notebook assigns binary protein labels — **High Protein (1)** and **Low Protein (0)** — to each of the 80 corn samples using a median split on the Protein column.

The labeled dataset is saved to `data/processed/labeled.csv` for use in all downstream notebooks.

## Section 1 — Imports and Setup

We import all required Python libraries here. We also set a consistent visual style for all plots and define a `RANDOM_STATE = 42` for reproducibility throughout the project.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import seaborn as sns

# Fixed random state for reproducibility across the project
RANDOM_STATE = 42

# Clean, publication-ready plot style
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

print('Libraries loaded successfully.')

## Section 2 — Load the Dataset

We load `data/raw/corn_mp5_regression_data.csv` — the raw Eigenvector Research corn NIR dataset.

Each row corresponds to one corn sample. The columns are:
- **SampleID** — unique sample identifier
- **Wave_1 to Wave_700** — NIR absorbance values at 700 discrete wavelengths
- **Moisture, Starch, Oil, Protein** — chemical composition labels measured by wet chemistry

We then separate the data into three parts:
- `X` → the 700 spectral columns as a NumPy array (the input features)
- `protein` → the Protein column as a pandas Series (the quantity we will threshold)
- `sample_ids` → the SampleID column as a Python list

In [ ]:
# Load the raw dataset
df = pd.read_csv('../data/raw/corn_mp5_regression_data.csv')

print('Dataset shape:', df.shape)
df.head()

In [ ]:
# Identify the 700 spectral columns
spectral_cols = [f'Wave_{i}' for i in range(1, 701)]

# Separate into spectra matrix, protein series, and sample ID list
X          = df[spectral_cols].to_numpy()
protein    = df['Protein']
sample_ids = df['SampleID'].tolist()

print(f'X shape (spectra matrix): {X.shape}')
print(f'protein series shape:     {protein.shape}')
print(f'Number of sample IDs:     {len(sample_ids)}')
print(f'First 5 sample IDs:       {sample_ids[:5]}')

## Section 3 — Compute the Median Threshold

To convert continuous Protein values into binary class labels, we need a threshold. We use the **median** of the protein values as the cut-off point, for the following reasons:

1. **No universal industry standard exists for this specific measurement scale.**  
   The protein percentages in this dataset are calibrated to a specific NIR instrument setup. They cannot be directly compared to general agricultural grade thresholds (e.g., USDA commodity standards) without domain-specific recalibration knowledge. Using an external hardcoded threshold would be arbitrary and scientifically unjustifiable.

2. **The median guarantees a balanced class split.**  
   For 80 samples, the median naturally divides the dataset into two equal or near-equal groups. Balanced classes lead to fairer model training and more reliable performance metrics — imbalanced classes can cause a classifier to simply predict the majority class.

3. **It is fully data-driven and reproducible.**  
   The threshold is derived entirely from the dataset itself, making it transparent and defensible in a thesis context. Any reviewer can verify it by computing `protein.median()` from the same CSV file.

Samples with protein ≥ median → **High Protein (Label = 1)**  
Samples with protein < median → **Low Protein (Label = 0)**

In [ ]:
# Compute the median protein value across all 80 samples
protein_median = protein.median()
print(f'Median protein value: {protein_median:.4f}%')

# Count samples on each side of the threshold
above = (protein >= protein_median).sum()
below = (protein <  protein_median).sum()

print(f'Samples >= median (will be High Protein): {above}')
print(f'Samples <  median (will be Low Protein):  {below}')

## Section 4 — Assign Binary Labels

We create a new column `Protein_Label` in the dataframe with the following encoding:

- **1 = High Protein** — samples whose Protein value is **greater than or equal to** the median
- **0 = Low Protein** — samples whose Protein value is **strictly less than** the median

This binary encoding (**1 = High Protein, 0 = Low Protein**) is the convention used consistently throughout the **entire project** — in all preprocessing, splitting, augmentation, model training, and evaluation notebooks.

In [ ]:
# Assign binary labels: 1 = High Protein, 0 = Low Protein
df['Protein_Label'] = (protein >= protein_median).astype(int)

print('Protein_Label value counts:')
print(df['Protein_Label'].value_counts().sort_index())
print()
print('Encoding: 0 = Low Protein, 1 = High Protein')

## Section 5 — Visualize the Labeled Distribution

We plot the distribution of Protein values as a histogram with a KDE (Kernel Density Estimate) overlay. The two groups are shown in different colors — blue for Low Protein and red for High Protein — and a vertical dashed line marks the median threshold.

In [ ]:
high_mask = df['Protein_Label'] == 1
low_mask  = df['Protein_Label'] == 0

fig, ax = plt.subplots(figsize=(10, 5))

# Plot each protein group separately using seaborn
sns.histplot(protein[low_mask],  color='#3498db', kde=True, ax=ax, alpha=0.6)
sns.histplot(protein[high_mask], color='#e74c3c', kde=True, ax=ax, alpha=0.6)

# Draw the median threshold as a vertical dashed line
ax.axvline(protein_median, color='darkred', linestyle='--', linewidth=2)

# Build a clean legend manually so the median line is included
low_patch   = mpatches.Patch(facecolor='#3498db', alpha=0.6, label='Low Protein (0)')
high_patch  = mpatches.Patch(facecolor='#e74c3c', alpha=0.6, label='High Protein (1)')
median_line = mlines.Line2D([], [], color='darkred', linestyle='--', linewidth=2,
                            label=f'Median = {protein_median:.3f}')

ax.set_title('Protein Content Distribution with High/Low Labels')
ax.set_xlabel('Protein Content (%)')
ax.set_ylabel('Count')
ax.legend(handles=[low_patch, high_patch, median_line])
plt.tight_layout()
plt.show()

## Section 6 — Visualize Spectra by Label

We plot all 80 NIR spectra on the same set of axes, colored by their assigned protein label. **Red** lines represent High Protein samples; **blue** lines represent Low Protein samples. A low alpha of 0.3 is used so that overlapping spectra remain visible.

The x-axis is the actual wavelength scale of the Foss NIR Systems 6500 instrument used to collect this dataset: **1100 nm to 2498 nm at 2 nm intervals**, giving exactly 700 wavelength points.

In [ ]:
# Wavelength axis: 1100 nm to 2498 nm with 2 nm steps → 700 points
wavelengths = np.arange(1100, 2499, 2)
print(f'Wavelength axis: {wavelengths[0]} nm to {wavelengths[-1]} nm, {len(wavelengths)} points')

fig, ax = plt.subplots(figsize=(12, 5))

high_plotted = False
low_plotted  = False

for i in range(len(X)):
    if df['Protein_Label'].iloc[i] == 1:
        # Only add to legend once per class
        lbl = 'High Protein (1)' if not high_plotted else '_nolegend_'
        ax.plot(wavelengths, X[i], color='red',  alpha=0.3, label=lbl)
        high_plotted = True
    else:
        lbl = 'Low Protein (0)' if not low_plotted else '_nolegend_'
        ax.plot(wavelengths, X[i], color='blue', alpha=0.3, label=lbl)
        low_plotted = True

ax.set_title('NIR Spectra Colored by Protein Label')
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Absorbance')
ax.legend()
plt.tight_layout()
plt.show()

## Section 7 — Save the Labeled Dataset

We assemble the final labeled dataframe containing:
- The **SampleID** column
- All **700 wavelength columns** (`Wave_1` to `Wave_700`)
- The four original label columns: **Moisture, Starch, Oil, Protein**
- The new **Protein_Label** column (binary: 0 or 1)

This file is saved to `data/processed/labeled.csv` and is the starting point for all future notebooks.

In [ ]:
# Assemble the output dataframe in the correct column order
output_cols = ['SampleID'] + spectral_cols + ['Moisture', 'Starch', 'Oil', 'Protein', 'Protein_Label']
df_labeled  = df[output_cols].copy()

# Save to data/processed/labeled.csv
output_path = '../data/processed/labeled.csv'
df_labeled.to_csv(output_path, index=False)

print(f'File saved successfully to: {output_path}')
print(f'Shape of saved dataframe:   {df_labeled.shape}')

## Section 8 — Summary

The following table summarizes everything accomplished in this notebook:

| Item | Details |
|------|---------|
| **Median protein threshold** | Computed from 80 samples — exact value printed in Section 3 |
| **High Protein samples (Label = 1)** | Samples with Protein ≥ median — count printed in Section 4 |
| **Low Protein samples (Label = 0)** | Samples with Protein < median — count printed in Section 4 |
| **Output file** | `data/processed/labeled.csv` |

The median split produces a **balanced binary classification dataset**, which is essential for fair model training and reliable evaluation metrics.

The label encoding **1 = High Protein** and **0 = Low Protein** is used consistently throughout the entire project.

---

### What Comes Next?

The next notebook, **`03_sg_preprocessing.ipynb`**, will apply **Savitzky-Golay smoothing** to the raw NIR spectral data in `data/processed/labeled.csv`. This preprocessing step reduces high-frequency instrument noise in the spectra while preserving the meaningful absorption features needed for accurate protein classification.